# Vaccine Tracker

- The first dataset is the 2019 US Census that has the total number of inhabitants of each state, along with latitude and longitude data for each state's capital city.
- The second dataset contains an estimate for the total number of people that have been vaccinated in each state at the time.

In the next code cell, we load and preprocess the data.  As output, we'll see the total percent of the population that has been vaccinated in the US, along with a preview of the Pandas DataFrame that we'll use to make the tracker.

In [1]:
# Imports
import pandas as pd
from datetime import date
import folium
from folium import Marker
from folium.plugins import MarkerCluster
import math
import matplotlib.pyplot as plt
import seaborn as sns

# Population Data
populationData = pd.read_csv('2019_Census_US_Population_Data_By_State_Lat_Long.csv')

# Vaccination data for the latest available date in the dataset
vaccinationData = pd.read_csv('us_state_vaccinations.csv')
vaccinationData['date'] = pd.to_datetime(vaccinationData['date'])

latestDate = vaccinationData['date'].max()
latestDateStr = latestDate.strftime('%Y-%m-%d')

vaccinationByLocation = vaccinationData.loc[
    vaccinationData['date'] == latestDate,
    ["location", "people_vaccinated"]
]

# Vaccination and population data
vaccinationAndPopulationByLocation = pd.merge(
    populationData,
    vaccinationByLocation,
    left_on='STATE',
    right_on='location'
).drop(columns='location')

# Calculate percentage vaccinated by state
vaccinationAndPopulationByLocation['percent_vaccinated'] = (
    vaccinationAndPopulationByLocation['people_vaccinated'] /
    vaccinationAndPopulationByLocation['POPESTIMATE2019']
)

print('Data date:', latestDateStr)
vaccinationAndPopulationByLocation

Data date: 2023-05-10


,STATE,POPESTIMATE2019,lat,long,people_vaccinated,percent_vaccinated
0,Alabama,4903185,32.377716,-86.300568,3193141.0,0.651238
1,Alaska,731545,58.301598,-134.420212,535718.0,0.732310
2,Arizona,7278717,33.448143,-112.096962,5704677.0,0.783748
3,Arkansas,3017804,34.746613,-92.288986,2115165.0,0.700895
4,California,39512223,38.576668,-121.493629,33613401.0,0.850709
5,Colorado,5758736,39.739227,-104.984856,4837792.0,0.840079
6,Connecticut,3565287,41.764046,-72.682198,3670090.0,1.029395
7,Delaware,973764,39.157307,-75.519722,861811.0,0.885031
8,District of Columbia,705749,38.895110,-77.036370,836680.0,1.185521
9,Florida,21477737,30.438118,-84.281296,17810446.0,0.829252


In [2]:
print('Data date:', latestDateStr)

# Calculate the total percent vaccinated in the US
percentageTotal = vaccinationAndPopulationByLocation["people_vaccinated"].sum() / vaccinationAndPopulationByLocation["POPESTIMATE2019"].sum()
print('Percentage Vaccinated in the US: {}%'.format(round(percentageTotal*100, 2))) 

Data date: 2023-05-10
Percentage Vaccinated in the US: 80.04%


The next code cell uses the data to create a tracker, with one marker for each state.  We can click on the markers to see the percentage of the population that has been vaccinated.

In [3]:
# Create the map for maker clusters where each marker is a state and the tooltip shows the percentage vaccinated in that state
v_map = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=4) 

# Add points to the map
mc = MarkerCluster()
for idx, row in vaccinationAndPopulationByLocation.iterrows(): 
    if not math.isnan(row['long']) and not math.isnan(row['lat']):
        mc.add_child(Marker(location=[row['lat'], row['long']],
                            tooltip=str(round(row['percent_vaccinated']*100, 2))+"%"))
v_map.add_child(mc)

# Display the map
v_map